# Lab 8b (optional): The same agent in CrewAI
**Module 4.2, framework comparison.**  **Time:** 15 minutes. Best run in **Google Colab**.

CrewAI is an *opinionated* framework: you describe **Agents** (role, goal, backstory), **Tasks** and a **Crew**,
and it runs the tool loop from Lab 8 for you. Compare what you write here with Lab 8.

**Install note:** CrewAI is large (about 750 MB with its dependencies). The install cell takes 1 to 2 minutes
in Colab. On a laptop, install it in a separate virtual environment from the main labs (see SETUP.md).

In [ ]:
# SETUP: run this cell first. It works in Google Colab and in VS Code.
import os, sys, subprocess
REPO_URL = "https://github.com/cybertide-solutions/ai-platform-architect-labs.git"
if "google.colab" in sys.modules:
    if not os.path.exists("/content/course"):
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/course"], check=True)
    os.chdir("/content/course")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")                      # VS Code starts in notebooks/; the course files are one level up
sys.path.insert(0, os.getcwd())
import labkit as lk
lk.setup();

In [ ]:
# Install CrewAI (skipped if already installed). Telemetry is switched off for the course.
import os
lk.pip_install("crewai", module="crewai")
os.environ["CREWAI_DISABLE_TELEMETRY"] = "true"
os.environ["OTEL_SDK_DISABLED"] = "true"
os.environ["CREWAI_TRACING_ENABLED"] = "false"

In [ ]:
from crewai import Agent, Task, Crew, LLM
from crewai.tools import tool

# CrewAI talks to the same provider and model labkit picked (Groq or Gemini), through its OpenAI-compatible API.
llm = LLM(model="openai/" + lk.model_name("big"), base_url=lk.CONFIG["base_url"],
          api_key=os.environ.get("GROQ_API_KEY") or os.environ.get("GEMINI_API_KEY") or "not-needed",
          temperature=0)

ORDERS = {"ORD-1001": {"status": "shipped", "amount_inr": 2499},
          "ORD-1002": {"status": "delivered", "amount_inr": 7999}}

@tool("order_status")
def order_status(order_id: str) -> str:
    """Get the status and amount of a customer order by its id, for example ORD-1001."""
    return str(ORDERS.get(order_id.upper(), "not found"))

In [ ]:
support = Agent(role="Customer support agent",
                goal="Answer customer questions about their orders accurately",
                backstory="You work for Meridian Retail and always check the order system before answering.",
                tools=[order_status], llm=llm, verbose=True)

task = Task(description="Customer asks: {question}",
            expected_output="A short, friendly answer that states the order status.",
            agent=support)

crew = Crew(agents=[support], tasks=[task], verbose=True)
# In a notebook use kickoff_async with await. Plain crew.kickoff() fails inside Jupyter/Colab with
# "Agent execution was invoked synchronously from within a running event loop". In a .py script, kickoff() is fine.
result = await crew.kickoff_async(inputs={"question": "Where is my order ORD-1001?"})
print("\nRESULT:", result)

## Compare with Lab 8
| Lab 8 (plain Python) | CrewAI |
|---|---|
| system prompt | `role`, `goal`, `backstory` |
| user message | `Task.description` |
| the `for step in range(...)` loop | `Crew.kickoff()` (`kickoff_async()` in notebooks) |
| `tool_spec(...)` | `@tool` decorator with a docstring |
| your approval gate | `human_input=True` on a Task, or a check inside the tool |

**Talking point:** frameworks save code and add structure (multi-agent roles, memory, tracing), but they also
add dependencies, abstractions and their own release cycle. The pattern underneath is the same in CrewAI,
Google ADK, LangGraph or the OpenAI Agents SDK.